# Sprout Imitation 4K Rendering

Loads the Sprout imitation checkpoint, generates rollouts for every clip in the
training dataset (Male2MartialArtsPunches, 18 clips), and renders 4K (3840×2160)
videos showing both the AMASS reference trajectory (transparent ghost) and the
imitator robot.

In [ ]:
%load_ext autoreload
%autoreload 2

import os

os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

import imageio
import jax
import jax.numpy as jp
import mediapy as media
import numpy as np
from pathlib import Path

from track_mjx.agent import checkpointing
from track_mjx.analysis import rollout, utils

## Configuration

In [ ]:
CKPT_PATH = Path(
    "/home/talmolab/Desktop/SalkResearch/track-mjx/model_checkpoints/"
    "260203_104343_943116"
)
OUTPUT_DIR = CKPT_PATH / "4k_renders"
OUTPUT_DIR.mkdir(exist_ok=True)

# 4K rendering settings
RENDER_WIDTH = 3840
RENDER_HEIGHT = 2160
RENDER_FPS = 50
CAMERA = "close_profile"

## Load Checkpoint

In [ ]:
ckpt = checkpointing.load_checkpoint_for_eval(str(CKPT_PATH))
cfg = ckpt["cfg"]

# Verify the data path exists
data_path = Path(cfg.env_config.reference_data_path)
print(f"Reference data: {data_path}")
print(f"Data exists: {data_path.exists()}")
print(f"Env name: {cfg.env_config.env_name}")
print(f"Clip length: {cfg.env_config.clip_length}")
print(f"Camera: {cfg.render_config.render_camera_name}")

In [ ]:
env = rollout.create_environment(cfg)
print(f"Action size: {env.action_size}")
print(f"Number of clips: {env.reference_clips.qpos.shape[0]}")
print(f"Clip shape (n_clips, n_frames, qpos_dim): {env.reference_clips.qpos.shape}")

n_clips = env.reference_clips.qpos.shape[0]
if env.reference_clips.has_variable_clip_lengths:
    clip_lengths = env.reference_clips.clip_lengths
    print(f"Variable clip lengths: {clip_lengths}")
else:
    print(f"Fixed clip length: {env.reference_clips.qpos.shape[1]}")

In [ ]:
inference_fn = checkpointing.load_inference_fn(cfg, ckpt["policy"])
generate_rollout = rollout.create_rollout_generator(
    cfg,
    env,
    inference_fn,
    log_full_states=True,
    log_activations=False,
    log_metrics=False,
    log_sensor_data=False,
)
print("Inference function and rollout generator ready.")

## Test: Single Clip 4K Render

Run one clip first to verify the pipeline works before rendering all clips.

In [ ]:
# Generate rollout for first clip
test_rollout = generate_rollout(clip_idx=0)
test_states = test_rollout["rollout_states"]

# Convert stacked states to list for rendering
num_timesteps = test_states.data.qpos.shape[0]
test_states_list = [
    jax.tree_util.tree_map(lambda x: x[i], test_states)
    for i in range(num_timesteps)
]
print(f"Generated rollout with {num_timesteps} timesteps")
print(f"Mean reward: {float(test_rollout['state_rewards'].mean()):.4f}")

In [ ]:
# Render at 4K with ghost reference
test_frames = env.render(
    test_states_list,
    height=RENDER_HEIGHT,
    width=RENDER_WIDTH,
    camera=CAMERA,
    render_ghost=True,
    add_labels=True,
)
print(f"Rendered {len(test_frames)} frames at {test_frames[0].shape}")

In [ ]:
# Save test video
test_video_path = OUTPUT_DIR / "test_clip0_4k.mp4"
with imageio.get_writer(str(test_video_path), fps=RENDER_FPS) as writer:
    for frame in test_frames:
        writer.append_data(frame)
print(f"Saved test video: {test_video_path}")
print(f"File size: {test_video_path.stat().st_size / 1024 / 1024:.1f} MB")

In [ ]:
# Display inline (downscaled for notebook display)
media.show_video(test_frames, fps=RENDER_FPS)

## Render All Clips at 4K

Loop over all clips in the dataset. For each clip:
1. Generate a rollout using the trained policy
2. Render at 4K with the AMASS reference ghost overlay
3. Save as individual MP4 video

In [ ]:
import time

for clip_idx in range(n_clips):
    clip_start = time.time()

    # Get clip name if available
    if hasattr(env.reference_clips, "clip_names"):
        clip_name = str(env.reference_clips.clip_names[clip_idx])
    else:
        clip_name = f"clip{clip_idx:03d}"

    print(f"\n--- Clip {clip_idx}/{n_clips-1}: {clip_name} ---")

    # Generate rollout
    rollout_data = generate_rollout(clip_idx=clip_idx)
    states = rollout_data["rollout_states"]
    num_steps = states.data.qpos.shape[0]
    mean_reward = float(rollout_data["state_rewards"].mean())
    print(f"  Rollout: {num_steps} steps, mean reward: {mean_reward:.4f}")

    # Convert stacked states to list
    states_list = [
        jax.tree_util.tree_map(lambda x: x[i], states) for i in range(num_steps)
    ]

    # Render at 4K with ghost
    frames = env.render(
        states_list,
        height=RENDER_HEIGHT,
        width=RENDER_WIDTH,
        camera=CAMERA,
        render_ghost=True,
        add_labels=True,
    )

    # Save video
    video_path = OUTPUT_DIR / f"{clip_name}_4k.mp4"
    with imageio.get_writer(str(video_path), fps=RENDER_FPS) as writer:
        for frame in frames:
            writer.append_data(frame)

    elapsed = time.time() - clip_start
    file_size = video_path.stat().st_size / 1024 / 1024
    print(f"  Saved: {video_path.name} ({file_size:.1f} MB, {elapsed:.1f}s)")

In [ ]:
# Summary
print(f"\n{'='*60}")
print(f"Rendering complete!")
print(f"Output directory: {OUTPUT_DIR}")
videos = sorted(OUTPUT_DIR.glob("*.mp4"))
total_size = sum(v.stat().st_size for v in videos) / 1024 / 1024
print(f"Total videos: {len(videos)}")
print(f"Total size: {total_size:.1f} MB")
for v in videos:
    print(f"  {v.name}: {v.stat().st_size / 1024 / 1024:.1f} MB")